In [2]:
##Note this is a version of the original code that has been edited to remove
##confidential information. Thus, it is for example only and does not run

#Load Libraries

# IMPORTANT TO GET VERSIONS RIGHT:
#!pip install selenium==3.141.0
#!pip install appium-python-client==1.3.0

from cgi import print_exception
from appium import webdriver
import time
import pandas as pd
import glob
from pathlib import Path
from selenium.webdriver.common.by import By
import os
import pyautogui
from selenium.webdriver.common.keys import Keys
from time import sleep
from datetime import datetime
import math

print("Done")

Done


In [9]:
#Set File and folder locations and define general variables

#Coordinates
#Coordinates of the down arrow in folder path in Import an Organism screen
geometry_folder_x = 1301
geometry_folder_y = 141

#Coordinates of "Input Source" tab
input_source_tab_x = 995
input_source_tab_y = 367

#Coordinates of "Input" tab
input_tab_x = 1199
input_tab_y = 368

#Coordinates of "Tissue Concentrations" sub-tab
tissue_concentrations_x = 1102
tissue_concentrations_y = 401
    
#Coordinates of the down arrow in folder path in Save As window
save_folder_x = 1312
save_folder_y = 144

#RESRAD application location
RESRAD = "C:\RESRAD_Family\BIOTA\BIOTA1.8\Biota.exe"

tissue_input_file = ""

#Filename of log report of if you run terrestrial or aquatic for each subarea
log_filename = ""

#Filename of file with other file names for subareas that you want to run
select_runs_filename = "Selected_RESRAD_Runs.xlsx"

#Filename of file with geometries for each subarea to run
which_geometries_filename = "Taxa_per_Subarea_Summary.xlsx"

abiotic_input_folder = ""

output_folder = ""

environment_folder = "RESRAD Saved Environments"
environment_folder_typing = r"RESRAD Saved Environments"

geometries_folder_typing = r"Organism Geometry Files"

#Number of digits to enter (it takes longer to enter all of the digits)
num_digits = 3 #Can also be "All"

#Read in file that has info about run parameters
log = pd.read_csv(log_filename)

#Read in file that has which subareas you want to run
select_runs = pd.read_excel(select_runs_filename)

which_geometries = pd.read_excel(which_geometries_filename)

all_tissue_data = pd.read_excel(tissue_input_file)

time_tracker_filename = "RESRAD_Automation_Tracker.xlsx"

print("Done")


Done


In [4]:
# Define functions to connect to drivers
def create_resrad_driver():
    desired_caps = {}
    desired_caps['app'] = RESRAD
    return webdriver.Remote(command_executor='http://127.0.0.1:4723', desired_capabilities=desired_caps)

def create_desktop_driver():
    desktop_caps = {}
    desktop_caps["platformName"] = "Windows"
    desktop_caps["deviceName"] = "WindowsPC"
    desktop_caps["app"] = "Root"
    return webdriver.Remote(command_executor='http://127.0.0.1:4723', desired_capabilities=desktop_caps)

def create_save_as_driver(driver_desktop):
    save_as_window = driver_desktop.find_element(By.NAME, "Save As") #Identifies the Save As dialog box
    save_as_handle = save_as_window.get_attribute("NativeWindowHandle")
    save_as_handle = hex(int(save_as_handle))
    save_as_caps = {
        "appTopLevelWindow": save_as_handle
    }
            
    save_as_driver = webdriver.Remote(
        command_executor='http://127.0.0.1:4723',
        desired_capabilities=save_as_caps
    ) #Connects to Save as Dialog box to drive
    return save_as_driver
    
print("Done")

Done


In [10]:
def resrad_input():
    #Get current date and time
    resrad_driver = create_resrad_driver()
    resrad_driver.find_element_by_name("OK").click()
    
    #Create dataframe to collect summary data
    summary_df = pd.DataFrame(columns=["Subarea", "Terrestrial_Animal", "Terrestrial_Plant", "Aquatic_Animal", "Riparian_Animal"])
    
    for index, row in select_runs.iterrows():
        now = datetime.now()
        current_date = now.date()
        start_time = now.strftime("%H:%M:%S")
    
        filename = row['File_name']
        Area_name = row['Area_name']
        canyonMesa = row['CanyonMesa']
        watershed = row['Watershed']
        print('Area name:',Area_name)
        if canyonMesa == "Rio Grande":
            dataframe = pd.read_excel(abiotic_input_folder + "/" + canyonMesa + "/" + filename + '.xlsx')
        else:
            dataframe = pd.read_excel(abiotic_input_folder + "/" + canyonMesa + "/" + watershed + "/" + filename + '.xlsx')
        
        if Area_name in log['Area_name'].values:
            run_types = log.loc[log['Area_name']==Area_name,'RESRAD_Run'].iloc[0]
        elif canyonMesa == "Rio Grande":
            run_types = "Aquatic"
        else:
            run_types = "Terrestrial"
            
        level = str(which_geometries.loc[which_geometries['Area_Name']==Area_name,'TIER_RUN'].iloc[0])
        print('Level:',level)
        
        #Go to the next file if the dataframe does not have data
        if dataframe.empty: 
            print("no data: ", name)
            continue
            
        if run_types == "Both":
            run_types = ['Terrestrial','Aquatic']
        else:
            run_types = [run_types]
        print(run_types)
        
        for sub_index, run_type in enumerate(run_types):
            print(run_type)
            resrad_driver.find_element_by_accessibility_id("60").send_keys(Area_name)
            
            #Choose tier 3 to add tissue
            if level == "3": 
                resrad_driver.find_element_by_accessibility_id("57").click()
            elif level == "2":
                resrad_driver.find_element_by_accessibility_id("58").click()
            else:
                print("Unknown Level")
                continue
            
            #Click aquatic 
            resrad_driver.find_element_by_name("Aquatic").click()
            
            if run_type == "Terrestrial":
                #Edit sediment value to 0
                resrad_driver.find_element_by_accessibility_id("40").send_keys('0')
                #Click terrestrial
                resrad_driver.find_element_by_name("Terrestrial").click()
            
            #Set AM-241 indicator to zero (this will later be used to remove AM-241 from list if it's not needed)
            indicator = 0

            #Go row by row in the file to enter contaminant data
            for ind in dataframe.index:
                parameter = dataframe['PARAMETER_CODE'][ind]
                result_S = dataframe['Mean_Result_S'][ind]
                result_W = dataframe['Mean_Result_W'][ind]
                if(num_digits != "All"):
                    result_S = round(result_S,num_digits)
                    result_W = round(result_W,num_digits)
                if parameter == "Am-241" and math.isnan(result_W):
                    result_W = 0
                result_S = str(result_S)
                result_W = str(result_W)

                if parameter == 'Am-241':
                    resrad_driver.find_element_by_name('Am-241').click()
                    resrad_driver.find_element_by_accessibility_id("41").send_keys(result_S)
                    resrad_driver.find_element_by_accessibility_id("39").send_keys(result_W)
                    resrad_driver.find_element_by_accessibility_id("43").click()
                    indicator = indicator + 1

                else:
                    #Choose contaminant
                    resrad_driver.find_element_by_name(parameter).click()
                    resrad_driver.find_element_by_accessibility_id("45").click()
                    
                    if run_type == "Terrestrial":
                        #Enter soil value
                        resrad_driver.find_element_by_accessibility_id("41").send_keys(result_S)
                    elif run_type == "Aquatic":
                        #Enter sediment value
                        resrad_driver.find_element_by_accessibility_id("40").send_keys(result_S)
                    
                    #Enter water value
                    resrad_driver.find_element_by_accessibility_id("39").send_keys(result_W)

                    #Check "mean" box
                    resrad_driver.find_element_by_accessibility_id("43").click()

                #Remove Am-241 if it was not in the list
                if indicator == 0:
                    resrad_driver.find_element_by_name('Am-241').click()
                    resrad_driver.find_element_by_accessibility_id("44").click()
            
            driver_desktop = create_desktop_driver()  # connect to desktop driver
            sleep(2)
            if level == "3": 
                #Enter tissue information
                resrad_driver.find_element_by_accessibility_id("29").click() #Edit button

                geometries = which_geometries.loc[which_geometries['Area_Name']==Area_name,'TAXA_SUBGROUP']

                tissue_window = driver_desktop.find_element(By.NAME, "Organism-Specific Parameters")

                for taxon in geometries:
                    print(taxon)

                    tissue_data = all_tissue_data[(all_tissue_data['TAXA_SUBGROUP']==taxon) & (all_tissue_data['Area_Name']==Area_name)]

                    #Determine the indices of abiotic data parameters that you have tissue data for
                    abiotic_Para_in_tissue = dataframe['PARAMETER_CODE'].isin(tissue_data['PARAMETER_CODE'])

                    driver_desktop.find_element_by_accessibility_id("78").click()
                    sleep(2)

                    pyautogui.click(x=geometry_folder_x,y=geometry_folder_y)
                    
                    sleep(1)

                    pyautogui.write(geometries_folder_typing,interval=0.01)
                    pyautogui.press("enter")
                    sleep(1)

                    filename_box = driver_desktop.find_element(By.ACCESSIBILITY_ID, "1148") #Find the enter file name box
                    filename_box.clear()
                    sleep(1)
                    filename_box.send_keys(taxon)
                    sleep(1)
                    pyautogui.press("enter")

                    pyautogui.click(x=input_source_tab_x, y=input_source_tab_y)

                    cells = tissue_window.find_elements(By.ACCESSIBILITY_ID, "12759")

                    cells[0].click()
                    
                    #Send arrow key to move right to 2nd column
                    for _ in range(4):
                        cells[0].send_keys(Keys.RIGHT)
                        sleep(0.1)

                    for ind in dataframe.index:
                        pyautogui.write("Yes", interval=0.05)   
                        pyautogui.press("down")

                    pyautogui.click(x=input_tab_x, y=input_tab_y)
                    pyautogui.click(x=tissue_concentrations_x, y=tissue_concentrations_y)

                    cells2 = tissue_window.find_elements(By.ACCESSIBILITY_ID, "12759")

                    cells2[3].click()

                    #Send arrow key to move right to 2nd column
                    for _ in range(4):
                        cells2[3].send_keys(Keys.RIGHT)
                        sleep(0.1)

                    #For ind in dataframe.loc[abiotic_Para_in_tissue].index:
                    for ind in dataframe.index:
                        if abiotic_Para_in_tissue[ind]:
                            param_code = dataframe.loc[ind, 'PARAMETER_CODE']
                            tissue_conc = tissue_data.loc[tissue_data['PARAMETER_CODE'] == param_code, 'Tissue_for_RESRAD']
                            tissue_conc = str(tissue_conc.iloc[0])
                        else:
                            tissue_conc = str(0)
                        pyautogui.write(tissue_conc, interval=0.05) 
                        pyautogui.press("down")

                tissue_window.find_element_by_name('Close').click()
            
            #Save environment
            output_name_env = run_type + "_environment_" + Area_name + ".bio"
            
            if Path(environment_folder + "/" + output_name_env).exists():
                os.remove((environment_folder + "/" + output_name_env))
            
            resrad_driver.find_element_by_name("File").click()
            resrad_driver.find_element_by_name("Save As").click()
            sleep(2) #Makes sure there is enough time for the save as window to open
            save_as_driver = create_save_as_driver(driver_desktop)
            
            pyautogui.click(x=save_folder_x,y=save_folder_y)
            
            pyautogui.write(environment_folder_typing,interval=0.05)
            pyautogui.press("enter")
            sleep(1)
            
            filename_box = save_as_driver.find_element(By.ACCESSIBILITY_ID, "1001") #Find the enter file name box
            filename_box.clear()
            filename_box.send_keys(output_name_env)
            
            save_button = save_as_driver.find_element(By.NAME, "Save") 
            save_button.click() #Clicks the save button
            sleep(1)
            
            print(resrad_driver.session_id)
            #Run
            print(resrad_driver.window_handles)
            
            resrad_driver.find_element_by_name('Run').click()
            #Close window
            sleep(20)
            
            while True:
                try:
                    #Try to find the element
                    close_button = resrad_driver.find_element_by_accessibility_id("36")
                    close_button.click()
                    break  #Exit the loop once found and clicked
                except NoSuchElementException:
                    #Element not found, wait and try again
                    print("Close button not found. Waiting 10 seconds...")
                    time.sleep(10)

            
            #Convert html file to dataframe
            html_df = pd.read_html(environment_folder + "/RESRAD-BIOTA Dose Report.htm")
            if level == "3":
                html_df = html_df[:-1] #When tissue is run it adds on some extra text at the bottom that you want to remove
                
            for i, table in enumerate(html_df):
                filename = output_folder + "/" + table.columns.get_level_values(0)[0] + "_" + Area_name + "_dose_report.csv"
                print(filename)
                
                table.to_csv(filename,index=False)
            #New file
            end_time = datetime.now().strftime("%H:%M:%S")
            
            #Create a DataFrame with the new row
            new_row = pd.DataFrame([{'Date': current_date, 'StartTime': start_time, 'EndTime':end_time}])

            #Read existing data
            existing_data = pd.read_excel(time_tracker_filename)

            #Append the new row
            updated_data = pd.concat([existing_data, new_row], ignore_index=True)

            #Save back to Excel
            updated_data.to_excel(time_tracker_filename, index=False)
            resrad_driver.find_element_by_name("File").click()
            resrad_driver.find_element_by_name("New").click()
            
if __name__ == '__main__':
    print(os.getcwd())
    resrad_input()

print("Done")

C:\Users\kwheeler\Documents\LANL\RESRAD Automation\EXAMPLE RUN
Area name: Los_Alamos_Canyon_SANI
Level: 2
['Terrestrial', 'Aquatic']
Terrestrial
BCE5173A-380A-471E-AFD7-EF94F13C5991
['0x002105E8']
//iec.local/nmm/Los Alamos Assessment/04 Assessment Activities/TO3c_Eco Injury Quant/09_Biota_Exploration/19_RESRAD Tissue/Dose Report Outputs/San I/Terrestrial Animal_Los_Alamos_Canyon_SANI_dose_report.csv
//iec.local/nmm/Los Alamos Assessment/04 Assessment Activities/TO3c_Eco Injury Quant/09_Biota_Exploration/19_RESRAD Tissue/Dose Report Outputs/San I/Terrestrial Plant_Los_Alamos_Canyon_SANI_dose_report.csv
Aquatic
BCE5173A-380A-471E-AFD7-EF94F13C5991
['0x002105E8']
//iec.local/nmm/Los Alamos Assessment/04 Assessment Activities/TO3c_Eco Injury Quant/09_Biota_Exploration/19_RESRAD Tissue/Dose Report Outputs/San I/Aquatic Animal_Los_Alamos_Canyon_SANI_dose_report.csv
//iec.local/nmm/Los Alamos Assessment/04 Assessment Activities/TO3c_Eco Injury Quant/09_Biota_Exploration/19_RESRAD Tissue/Dose

In [3]:
##Use to identify coordinates on screen for above code block
### import time

print("Tracking mouse position every 1 second for 30 seconds...\n")

for i in range(30):
    x, y = pyautogui.position()
    print(f"Time {i + 1}s: Mouse at ({x}, {y})")
    time.sleep(1)  # Wait 1 second

Tracking mouse position every 1 second for 30 seconds...

Time 1s: Mouse at (7422, 951)
Time 2s: Mouse at (7422, 951)
Time 3s: Mouse at (346, 67)
Time 4s: Mouse at (537, 84)
Time 5s: Mouse at (537, 84)
Time 6s: Mouse at (537, 84)
Time 7s: Mouse at (537, 84)
Time 8s: Mouse at (537, 84)
Time 9s: Mouse at (537, 84)
Time 10s: Mouse at (537, 84)
Time 11s: Mouse at (537, 84)
Time 12s: Mouse at (537, 84)
Time 13s: Mouse at (537, 84)
Time 14s: Mouse at (537, 84)
Time 15s: Mouse at (537, 84)
Time 16s: Mouse at (537, 84)
Time 17s: Mouse at (537, 84)
Time 18s: Mouse at (537, 84)
Time 19s: Mouse at (802, 528)
Time 20s: Mouse at (851, 663)
Time 21s: Mouse at (1895, 11)
Time 22s: Mouse at (4542, 651)
Time 23s: Mouse at (7058, 905)
Time 24s: Mouse at (7232, 698)
Time 25s: Mouse at (6264, 794)
Time 26s: Mouse at (6548, 575)
Time 27s: Mouse at (6384, 1080)
Time 28s: Mouse at (6674, 818)
Time 29s: Mouse at (6371, 735)
Time 30s: Mouse at (7781, 1139)
